1. Libraries and initialising 
Change file path here

In [ ]:
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import savgol_filter
from pyproj import Transformer
import requests
import rasterio
import io
import math
import time

IMU_FILE_PATH = r"D:\Data_gathered\2026_04_28\Camera\17_30_00\back_28_04_2026-17_30_00.mcap"
GPS_FILE_PATH = r"D:\Data_gathered\2026_04_28\Rosbag\17_30_00\rosbag\rosbag_0.mcap"

Reading data

In [ ]:
GPS_LAT_LON_TOPIC = "/navsat_topic"
#TODO :Check if time is correct
IMU_TOPICS = "/zed/pose"

# ------------------- DATA CONTAINERS -----------------------
data = {"t": [], "x": [], "y": [], "z": []}
gps_data = {"t": [], "v": []}
gps_coords = {"t": [], "lat": [], "lon": []}

# ------------------- FILE READING --------------------------
print("Reading IMU and GPS files...")
# (IMU Reading)
with open(IMU_FILE_PATH, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=[IMU_TOPICS]):
        if channel.topic == IMU_TOPICS:
            data["t"].append(ros_msg.header.stamp.nanosec / 1e9 + ros_msg.header.stamp.sec)
            data["x"].append(ros_msg.pose.position.x)
            data["y"].append(ros_msg.pose.position.y)
            data["z"].append(ros_msg.pose.position.z)
print("Done with IMU")
# (GPS Reading)
with open(GPS_FILE_PATH, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=[GPS_LAT_LON_TOPIC]):
        # Get Coordinates
        if channel.topic == GPS_LAT_LON_TOPIC:
            gps_coords["t"].append(message.publish_time / 1e9)
            gps_coords["lat"].append(ros_msg.latitude)
            gps_coords["lon"].append(ros_msg.longitude)
print("Finished reading files")

Helper functions

In [ ]:
def quaternion_to_pitch(qx, qy, qz, qw): #TODO KNOW THIS
    qx, qy, qz, qw = np.array(qx), np.array(qy), np.array(qz), np.array(qw)
    sinp = 2 * (qw * qy - qz * qx)
    sinp = np.clip(sinp, -1, 1)
    return np.degrees(np.arcsin(sinp))

def get_dtm_elevation(x, y, session):
    """Fetches AHN4 DTM elevation for a single Dutch RD coordinate."""
    wcs_url = "https://service.pdok.nl/rws/ahn/wcs/v1_0"
    params = {
        "service": "WCS", "version": "1.0.0", "request": "GetCoverage",
        "coverage": "dtm_05m", "crs": "EPSG:28992",
        "bbox": f"{x-0.5},{y-0.5},{x+0.5},{y+0.5}",
        "width": 1, "height": 1, "format": "GEOTIFF"
    }
    try:
        resp = session.get(wcs_url, params=params, timeout=5)
        if resp.status_code == 200:
            with rasterio.open(io.BytesIO(resp.content)) as src:
                return src.read(1)[0][0]
    except Exception:
        pass
    return None

Height map processing

In [ ]:
print("Processing AHN4 True Ground Elevation...")

# Convert all GPS to RD coordinates 
transformer = Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True)
rd_xs, rd_ys = transformer.transform(gps_coords["lon"], gps_coords["lat"])

dtm_elevations = []
dtm_distances = []
current_distance = 0.0

# Smart Downsampling: Only query API if vehicle moved > 3 meters to avoid API bans
session = requests.Session()
last_x, last_y = None, None

print(f"Total GPS points: {len(rd_xs)}. Downsampling for API queries...")

for i in range(len(rd_xs)):
    x, y = rd_xs[i], rd_ys[i]
    
    if last_x is None:
        dist_step = 0
    else:
        dist_step = math.hypot(x - last_x, y - last_y)

    # Only query if we moved 3 meters from the last queried point
    if last_x is None or dist_step >= 3.0:
        h = get_dtm_elevation(x, y, session)
        if h is not None and -100 < h < 200:            
            dtm_elevations.append(h)
            dtm_distances.append(current_distance)
        
        last_x, last_y = x, y
        print(f"Queried DTM: {len(dtm_elevations)} points...", end="\r")
        time.sleep(0.05) # Polite delay for PDOK servers
    
    # Keep track of distance based on coordinates
    if i > 0:
        current_distance += math.hypot(x - rd_xs[i-1], y - rd_ys[i-1])

print("\nDone fetching DTM data.")


IMU PROCESSING


In [ ]:
d = data[IMU_TOPICS[0]]
raw_t = np.array(d["t"])
t_plot = raw_t - raw_t[0]

t_gps_raw = np.array(gps_data["t"])
v_gps = np.array(gps_data["v"])

#Distance calculation
v_gps_interp = np.interp(raw_t, t_gps_raw, v_gps) # Interpolated GPS speed at IMU timestamps
speed = np.abs(v_gps_interp) # Use absolute speed to ensure we get positive values for distance calculation

gps_dist_steps = np.sqrt(np.diff(rd_xs)**2 + np.diff(rd_ys)**2) # Calculate distance steps from GPS coordinates
gps_cum_dist = np.insert(np.cumsum(gps_dist_steps), 0, 0.0) # Start at 0m

imu_distance = np.interp(raw_t, gps_coords["t"], gps_cum_dist) # Interpolate GPS cumulative distance at IMU timestamps
ds = np.diff(imu_distance, prepend=0)
# Pitch calculation
pitch = quaternion_to_pitch(d["qx"], d["qy"], d["qz"], d["qw"])
pitch_bias = np.mean(pitch)
print(f"Calculated pitch bias: {pitch_bias:.2f} degrees")

pitch = (pitch - pitch_bias) # Remove bias (and invert sign) to match DTM convention
#pitch = savgol_filter(pitch, window_length=101, polyorder=3) 


# IMU Estimated Elevation
dt = np.diff(t_plot, prepend=0)
imu_elevation = np.cumsum(ds * np.sin(np.radians(pitch)))
# Align IMU elevation to start at the exact same NAP height as the DTM
if len(dtm_elevations) > 0:
    imu_elevation = imu_elevation + dtm_elevations[0]

PLOT

In [ ]:
# --- PLOTTING ---
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.plot(imu_distance, imu_elevation, label="IMU Estimated Elevation", color='blue', linewidth=2)

# Plot DTM Ground Truth
if len(dtm_distances) > 0:
    ax.plot(dtm_distances, dtm_elevations, label="AHN4 DTM (True Ground)", color='green', linewidth=2, linestyle="--")

ax.set_title("Ground Profile: IMU Estimate vs AHN4 True DTM", fontsize=14, fontweight="bold")
ax.set_xlabel("Distance Traveled (m)")
ax.set_ylabel("Elevation (m NAP)")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

## FROM HERE

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.plot(data["t"], data["z"], label="IMU Z Position", color='blue', linewidth=2)
ax.set_title("IMU Z Position Over Time", fontsize=14, fontweight="bold")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Z Position (m)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
TIME = 1777390454.143338022

In [ ]:
print(data["t"][0], data["t"][-1])
t = np.array(data["t"])
x = np.array(data["x"])
y = np.array(data["y"])
z = np.array(data["z"])

ref_idx = np.argmin(np.abs(t - TIME)) #argmin returns the index of the minimum value in the array, which corresponds to the closest timestamp to TIME
dt = abs(data["t"][ref_idx] - TIME)
print(f"Using TIME = {TIME}, closest GPS message Δt = {dt:.3f}s " f"(index {ref_idx})")
print(len(t))


s = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
cum_s = np.concatenate([[0], np.cumsum(s)])
cum_s_from_s0 = cum_s - cum_s[ref_idx]
idx_25m = np.argmax(cum_s_from_s0 >= 25.0)
mask = (t >= t[ref_idx]) & (t <= t[idx_25m])
s_rel = cum_s[mask]
z_rel = z[mask] - z[ref_idx]
t_rel = t[mask]
print(cum_s)

plt.figure(figsize=(10, 5))
plt.plot(t_rel, s_rel, label="IMU Z relative to 25m point")
plt.xlabel("Distance from 25m point (m)")
plt.ylabel("Z relative to 25m point (m)")   
plt.title("IMU Z Position Relative to 25m Point")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

